# Phase 6 - Abstractive summarization with Arch 4 - DeepSeek-style MoE decoder

Starts from the Phase-1 model (12-layer MoE decoder (222.6M total, ~81M active): MLA attention, 32 routed + 1 shared expert per layer.) and writes a summary of each judgment.
Metric: BERTScore F1 (plus ROUGE-1 / ROUGE-L). The model reads `[source] <eos> [summary]` and is trained only on the summary tokens; at test time it continues after `<eos>` with a KV cache.

**What the data asks for.** Reference summaries are long (median ~640 words, 90% under 1,500) and about **4.6 times
shorter than their document**; documents run from ~1,300 to 140,000 words.
* **Source selection**: 1,000 tokens - 40% from the opening of the judgment (parties, facts) plus three evenly spaced
  pieces of the rest, the last ending at the final word, so the whole case is represented.
* **Length control**: each summary's length budget follows its document's length (document words / 4.6, clipped to
  150-900 words); generation has to reach 85% of it and stops by 115% - length matters a lot for BERTScore and ROUGE.
* **Redundancy control**: greedy decoding with a repetition penalty (Keskar et al. 2019) and no repeated 4-grams.

**Experiments** (each is its own submission; all from the Phase-6 list)

| Experiment | What trains |
|---|---|
| `full` | every weight - strong reference and the distillation teacher |
| `lorma` | **LoRMA** (compulsory): `(alpha/r BA + I) W x` on the attention projections (Bihany, Patel & Modi 2025) |
| `lora` | LoRA on the same projections (Hu et al. 2021) |
| `houlsby` | Houlsby bottleneck adapters after the attention and feed-forward outputs (Houlsby et al. 2019) |
| `prompt` | prompt tuning: 16 learned virtual tokens in front of the source (Lester et al. 2021) |
| `kd` | word-level distillation (Hinton et al. 2015; Kim & Rush 2016): a half-depth student learns the fine-tuned `full` model's next-token distributions |

There is no dev set, so 150 training documents are held out: teacher-forced loss on them picks the best checkpoint,
and 40 of them are actually summarised to report ROUGE-1 / ROUGE-L and length. Every experiment trains for a fixed
**time budget** (`TRAIN_MINUTES`).

**Kaggle settings:** GPU T4 x2, Internet on, secret `HF` holding your Hugging Face token. Then Run All.

## 1. Setup

In [1]:
import os, sys, json, glob, shutil, time, gc
import numpy as np
import torch
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

WORK_DIR = "/kaggle/working/phase6_arch4"
os.makedirs(WORK_DIR, exist_ok=True)
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


GPUs: ['Tesla T4', 'Tesla T4']


## 2. Configuration
Everything you might want to change lives here.

In [2]:
%%writefile /kaggle/working/phase6_arch4/config.py
"""Every knob of this Phase-6 (abstractive summarization) notebook."""
import os
import json

ARCH = "arch4"
WORK_DIR = f"/kaggle/working/phase6_{ARCH}"
DATA_DIR = os.path.join(WORK_DIR, "data")
CACHE_DIR = os.path.join(WORK_DIR, "phase1")        # Phase-1 files downloaded from Hugging Face land here
OUT_DIR = os.path.join(WORK_DIR, "runs")             # one folder per experiment

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
RAW = {"train": os.path.join(DATA_DIR, "Phase_6_Summ", "Train", "train.json"),
       "test": os.path.join(DATA_DIR, "Phase_6_Summ", "Test", "test.json")}

# ------------------------------------------------------------------ the Phase-1 model
# Looked up in this order: an attached Kaggle dataset whose path mentions "arch4", then these HF repos.
# The 600M-token run (v2) is preferred; the earlier 100M-token run is the fallback.
PHASE1_REPOS = ["D-240357/legal-arch4-moe-v2", "D-240357/legal-arch4-moe"]
WEIGHT_FILES = ["model_final.pt", "model_best.pt"]
TOKENIZER_FILES = ["tokenizer.json", "data/tokenizer.json"]
TOKENIZER_REPOS = PHASE1_REPOS + ["D-240357/legal-arch3-decoder"]
KAGGLE_HINT = "arch4"

SUM_REPO = "D-240357/legal-sum-arch4"           # fine-tuned checkpoints, results and submission zips go here

# ------------------------------------------------------------------ experiments
# Each one is a separate submission. "full" is the strong reference and the distillation teacher;
# "kd" waits for it to finish. "pfeiffer" (adapters after the feed-forward block only) is available too.
METHODS = ["full", "lorma", "lora", "houlsby", "prompt", "kd"]
KD_TEACHER = "full"
TRAIN_MINUTES = 40             # wall-clock training budget per experiment (the LR schedule ends exactly on it)
MAX_EPOCHS = 3

# ------------------------------------------------------------------ input and output lengths
SRC_LEN = 1000                 # source tokens the model reads
HEAD_FRAC = 0.4                # 40% from the opening of the judgment (parties, facts) ...
N_SPREAD = 3                   # ... and 3 evenly spaced pieces of the rest, the last ending at the final token
TGT_LEN = 1024                # summary tokens used for training (90% of references are shorter than this)
CHAR_CAP = 400_000             # text beyond this is not even tokenized
MAX_NEW_TOKENS = 1024         # hard ceiling at generation time (the model's context limit)
LENGTH_RATIO = "auto"          # summary words = document words / ratio (auto = training median, ~4.6) ...
MIN_SUMMARY_WORDS = 150        # ... clipped to these bounds
MAX_SUMMARY_WORDS = 900
LENGTH_SLACK = 0.15            # generation must reach 85% of the target length and stops by 115%

# ------------------------------------------------------------------ decoding
REPETITION_PENALTY = 1.15      # Keskar et al. 2019
NO_REPEAT_NGRAM = 4            # never emit the same 4-gram twice
SENTENCE_CASE = False         # arch 1 has a lower-case vocabulary: re-capitalise sentence starts
GEN_BATCH = 4

# ------------------------------------------------------------------ optimisation
MICRO_DOCS = 1
GRAD_ACCUM = 8
WARMUP_STEPS = 50
MIN_LR_RATIO = 0.1
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.05
FULL_LR = 3e-5
ADAPTER_LR = 5e-4
ADAPTER_RANK = 8
LORA_ALPHA = 16
LORMA_ALPHA = 8
ADAPTER_DROPOUT = 0.05
ADAPTER_TARGETS = ["q_down", "q_up", "kv_down", "kv_up", "out"]       # LoRA / LoRMA: attention projections
HOULSBY_SITES = ["out", "down"]         # after the attention output and the feed-forward output
PFEIFFER_SITES = ["down"]       # after the feed-forward output only
ADAPTER_BOTTLENECK = 64
PROMPT_LEN = 16
PROMPT_LR = 1e-2

KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5                 # weight of the reference-summary loss; the rest goes to matching the teacher
STUDENT_LAYERS = [0, 2, 4, 6, 8, 10]

# ------------------------------------------------------------------ evaluation
DEV_DOCS = 150                 # held out from train (there is no official dev set)
DEV_LOSS_DOCS = 100            # teacher-forced loss on these picks the best checkpoint
DEV_GEN_DOCS = 40              # these are actually summarised to measure ROUGE and length
EVAL_EVERY_STEPS = 150
TRAIN_DOC_LIMIT = None         # set to e.g. 200 (and TRAIN_MINUTES = 3) for a quick smoke test
PREP_WORKERS = max(1, os.cpu_count() or 2)

SEED = 13

_override = os.environ.get("P6_CFG_OVERRIDE")
if _override:
    globals().update(json.loads(_override))

Writing /kaggle/working/phase6_arch4/config.py


## 3. Code
The Phase-1 tokenizer and model (unchanged, so the trained weights load), plus the summarization pieces.

In [3]:
%%writefile /kaggle/working/phase6_arch4/tokenizer.py
"""
Byte-level BPE tokenizer, written from scratch (stdlib only for the tokenizer logic).

How it works, in short:
  1. A regex chops text into "pre-tokens" (a word with its leading space, a run of digits,
     a run of punctuation, ...). Merges never cross these boundaries.
  2. Every pre-token is turned into raw UTF-8 bytes (ids 0..255), so *any* text can be
     encoded - there is no <unk> and no lowercasing.
  3. Training repeatedly merges the most frequent adjacent id pair into a new id.
     We train on the table of *unique* pre-tokens with their counts, which is what keeps
     it fast in pure Python.
  4. Encoding replays the learned merges (lowest rank first) per pre-token, and caches the
     result, so repeated words cost a single dict lookup.
"""
import re
import json
import heapq
from collections import Counter, defaultdict

# Close to the GPT-2 splitter, but built on plain `re` (no \p{L}), so it needs no extra package.
PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+",
    re.UNICODE,
)

EOS_TOKEN = "<|endoftext|>"
PAD_TOKEN = "<|pad|>"


class ByteBPE:
    def __init__(self, merges=None, vocab_size=32000):
        self.vocab_size = vocab_size
        self.merges = [tuple(m) for m in (merges or [])]      # merge i creates id 256 + i
        self.eos_id = vocab_size - 2
        self.pad_id = vocab_size - 1
        self._cache = {}
        self._rebuild_tables()

    # ------------------------------------------------------------- internals
    def _rebuild_tables(self):
        self.ranks = {pair: (i, 256 + i) for i, pair in enumerate(self.merges)}
        vocab = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            vocab.append(vocab[a] + vocab[b])
        self.vocab = vocab
        self._cache.clear()

    # -------------------------------------------------------------- training
    @classmethod
    def train(cls, texts, vocab_size=32000, log_every=2000):
        """Learn merges from an iterable of strings."""
        n_merges = vocab_size - 256 - 2          # leave room for the two special tokens
        word_counts = Counter()
        for text in texts:
            word_counts.update(PRETOKEN_RE.findall(text))
        print(f"  unique pre-tokens: {len(word_counts):,}")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_count = defaultdict(int)
        where = defaultdict(set)                 # pair -> indices of words that (may) contain it
        for wi, w in enumerate(words):
            f = freqs[wi]
            for pair in zip(w, w[1:]):
                pair_count[pair] += f
                where[pair].add(wi)

        # Max-heap with lazy invalidation: stale entries are fixed up when they surface.
        heap = [(-c, p) for p, c in pair_count.items()]
        heapq.heapify(heap)

        merges = []
        vocab = [bytes([i]) for i in range(256)]
        for m in range(n_merges):
            best = None
            while heap:
                neg, pair = heapq.heappop(heap)
                real = pair_count.get(pair, 0)
                if real <= 0:
                    continue
                if real != -neg:
                    heapq.heappush(heap, (-real, pair))
                    continue
                best = pair
                break
            if best is None:
                print(f"  ran out of pairs after {m} merges")
                break

            new_id = 256 + m
            merges.append(best)
            a, b = best
            vocab.append(vocab[a] + vocab[b])
            touched = {}
            for wi in where.pop(best, ()):
                w = words[wi]
                f = freqs[wi]
                # find the occurrences in this word (left to right, non-overlapping)
                if len(w) < 2:
                    continue
                i, out, changed = 0, [], False
                while i < len(w):
                    if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                        changed = True
                    else:
                        out.append(w[i])
                        i += 1
                if not changed:
                    continue                      # stale entry in `where`
                for pair in zip(w, w[1:]):        # take the old word's pairs away ...
                    pair_count[pair] -= f
                for pair in zip(out, out[1:]):    # ... and add the new word's pairs
                    pair_count[pair] += f
                    where[pair].add(wi)
                    touched[pair] = pair_count[pair]
                words[wi] = out
            for pair, c in touched.items():
                if c > 0:
                    heapq.heappush(heap, (-c, pair))
            pair_count.pop(best, None)

            if (m + 1) % log_every == 0:
                print(f"  merge {m + 1:,}/{n_merges:,}  last = {vocab[-1]!r}")

        return cls(merges, vocab_size)

    # -------------------------------------------------------------- encoding
    def _encode_word(self, word):
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) > 1:
            best_rank, best_pair, best_new = None, None, None
            for pair in zip(ids, ids[1:]):
                hit = ranks.get(pair)
                if hit is not None and (best_rank is None or hit[0] < best_rank):
                    best_rank, best_pair, best_new = hit[0], pair, hit[1]
            if best_pair is None:
                break
            a, b = best_pair
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(best_new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out
        return ids

    def encode(self, text):
        cache = self._cache
        if len(cache) > 1_000_000:               # keep RAM bounded on huge corpora
            cache.clear()
        out = []
        for word in PRETOKEN_RE.findall(text):
            ids = cache.get(word)
            if ids is None:
                ids = cache[word] = self._encode_word(word)
            out.extend(ids)
        return out

    def decode(self, ids):
        specials = (self.eos_id, self.pad_id)
        data = b"".join(self.vocab[i] for i in ids if i < len(self.vocab) and i not in specials)
        return data.decode("utf-8", errors="replace")

    # ----------------------------------------------------------- persistence
    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges,
                       "eos": EOS_TOKEN, "pad": PAD_TOKEN}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            blob = json.load(f)
        return cls(blob["merges"], blob["vocab_size"])


# ---------------------------------------------------------------------------------------------------
# Phase 2 addition: encode with character offsets, so predicted tokens can be mapped back to exact spans.
# ---------------------------------------------------------------------------------------------------
# Same splitter as above, except a run of punctuation is cut into single characters. Gold entities often end
# in the middle of runs like "Ors.[1]" or "Cr.P.C.,"; with this split 99.3% of gold spans start and end on a
# token boundary (98.1% without it). Every piece is still encoded with the Phase-1 merges.
NER_PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]|\s+(?!\S)|\s+",
    re.UNICODE,
)


class NerTokenizer(ByteBPE):
    split_punct = True

    def encode_with_offsets(self, text):
        """Returns ids and, per token, the [start, end) of its non-whitespace characters in `text`."""
        splitter = NER_PRETOKEN_RE if self.split_punct else PRETOKEN_RE
        ids, starts, ends = [], [], []
        for m in splitter.finditer(text):
            word, base = m.group(), m.start()
            toks = self._cache.get(word)
            if toks is None:
                toks = self._cache[word] = self._encode_word(word)
            # byte position -> character position inside this piece (pieces can contain multi-byte characters)
            char_at_byte = []
            for ci, ch in enumerate(word):
                char_at_byte += [ci] * len(ch.encode("utf-8"))
            b = 0
            for t in toks:
                n = len(self.vocab[t])
                cs, ce = char_at_byte[b], char_at_byte[b + n - 1] + 1
                b += n
                piece = word[cs:ce]
                s = base + cs + (len(piece) - len(piece.lstrip()))
                e = base + ce - (len(piece) - len(piece.rstrip()))
                if s >= e:
                    s = e = base + ce
                ids.append(t)
                starts.append(s)
                ends.append(e)
        return ids, starts, ends

Writing /kaggle/working/phase6_arch4/tokenizer.py


In [4]:
%%writefile /kaggle/working/phase6_arch4/phase1_model.py
"""
Phase-1 model code, unchanged except for the small switches marked "Phase 2", so the saved weights load as-is.
"""
"""
Arch 4: a small DeepSeek-V3-flavoured mixture-of-experts decoder, written from scratch.

Ideas borrowed from the DeepSeek papers (scaled down to ~217M total / ~80M active parameters):

  * MLA - Multi-head Latent Attention. Keys/values are squeezed through a small latent vector,
    so the KV cache holds `kv_lora + qk_rope` numbers per token instead of full K and V.
    RoPE lives on a small separate slice of each head ("decoupled RoPE") so the latent stays
    position-free.
  * DeepSeekMoE - many small routed experts plus one shared expert that every token uses.
  * Auxiliary-loss-free load balancing (V3) - a per-expert bias is added to the routing scores
    *only to pick the top-k*; after every optimizer step it is nudged up for under-used experts
    and down for over-used ones. Gate weights come from the un-biased sigmoid scores. A tiny
    sequence-wise balance loss is kept only as a safety net.
  * Multi-token prediction (MTP) - during training an extra light module predicts the token two
    steps ahead. It shares the embedding / output head and is simply unused at inference.

Our own adaptation: expert dispatch uses fixed-size capacity buffers and one batched matmul over
all experts. Every tensor shape is static, so torch.compile can fuse it and there is no Python
loop over experts. In eval / generation the capacity is raised so that no token is ever dropped.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


@dataclass
class ModelConfig:
    vocab_size: int = 32000
    n_layer: int = 12
    n_head: int = 12
    d_model: int = 768
    max_seq_len: int = 2048
    rope_base: float = 10000.0
    norm_eps: float = 1e-5
    # MLA
    q_lora: int = 384
    kv_lora: int = 192
    qk_nope: int = 48
    qk_rope: int = 16
    v_dim: int = 64
    # MoE
    n_dense_layers: int = 1
    d_ff_dense: int = 2048
    n_routed: int = 32
    top_k: int = 4
    expert_ff: int = 192
    shared_ff: int = 512
    capacity_factor: float = 1.25
    # multi-token prediction
    mtp: bool = True
    mtp_ff: int = 1024


# ---------------------------------------------------------------------------
# small building blocks
# ---------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x32 * self.weight).to(x.dtype)


def build_rope_cache(dim, max_len, base):
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    angles = torch.outer(torch.arange(max_len).float(), inv_freq)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).to(x.dtype)


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.gate_up = nn.Linear(d_model, 2 * hidden, bias=False)
        self.down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        gate, up = self.gate_up(x).chunk(2, dim=-1)
        return self.down(F.silu(gate) * up)


# ---------------------------------------------------------------------------
# Multi-head Latent Attention
# ---------------------------------------------------------------------------
class MLA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.H = cfg.n_head
        self.nope, self.rope, self.vd = cfg.qk_nope, cfg.qk_rope, cfg.v_dim
        self.kv_lora = cfg.kv_lora

        self.q_down = nn.Linear(cfg.d_model, cfg.q_lora, bias=False)
        self.q_norm = RMSNorm(cfg.q_lora, cfg.norm_eps)
        self.q_up = nn.Linear(cfg.q_lora, self.H * (self.nope + self.rope), bias=False)

        # one projection yields the compressed KV latent AND the (shared) rotary key slice
        self.kv_down = nn.Linear(cfg.d_model, cfg.kv_lora + self.rope, bias=False)
        self.kv_norm = RMSNorm(cfg.kv_lora, cfg.norm_eps)
        self.kv_up = nn.Linear(cfg.kv_lora, self.H * (self.nope + self.vd), bias=False)

        self.out = nn.Linear(self.H * self.vd, cfg.d_model, bias=False)
        self.causal = True                         # Phase 2 turns this off to let tokens look right

    def forward(self, x, cos, sin, past=None, key_mask=None):
        B, T, _ = x.shape
        H = self.H

        q = self.q_up(self.q_norm(self.q_down(x))).view(B, T, H, self.nope + self.rope).transpose(1, 2)
        q_nope, q_rope = q.split([self.nope, self.rope], dim=-1)
        q_rope = apply_rope(q_rope, cos, sin)

        kv = self.kv_down(x)
        latent, k_rope = kv.split([self.kv_lora, self.rope], dim=-1)
        latent = self.kv_norm(latent)                                   # (B, T, kv_lora)
        k_rope = apply_rope(k_rope.unsqueeze(1), cos, sin)              # (B, 1, T, rope) shared by all heads

        if past is not None:                                            # cache = latent + rope key only
            latent = torch.cat([past[0], latent], dim=1)
            k_rope = torch.cat([past[1], k_rope], dim=2)
        present = (latent, k_rope)

        S = latent.size(1)
        kv_full = self.kv_up(latent).view(B, S, H, self.nope + self.vd).transpose(1, 2)
        k_nope, v = kv_full.split([self.nope, self.vd], dim=-1)

        q = torch.cat([q_nope, q_rope], dim=-1)
        k = torch.cat([k_nope, k_rope.expand(-1, H, -1, -1)], dim=-1)

        if key_mask is None:
            o = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal)
        else:
            o = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask)
        o = o.transpose(1, 2).reshape(B, T, H * self.vd)
        return self.out(o), present


# ---------------------------------------------------------------------------
# DeepSeekMoE layer with auxiliary-loss-free balancing and static-shape dispatch
# ---------------------------------------------------------------------------
@torch.compiler.disable
def _slot_positions(eid, E):
    """
    For every (token, expert) pair: its running position inside that expert's buffer, plus the
    one-hot assignment matrix. This is integer bookkeeping on ~33K elements, so it runs eagerly
    on purpose: comparing an index tensor with arange(E) trips an Inductor/Triton codegen bug on
    some torch builds (seen on Kaggle). Everything heavy around it stays compiled.
    """
    onehot = (eid.unsqueeze(1) == torch.arange(E, device=eid.device).unsqueeze(0)).to(torch.int32)
    pos = onehot.cumsum(0).gather(1, eid.unsqueeze(1)).squeeze(1) - 1
    return onehot, pos


class MoE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.E, self.K = cfg.n_routed, cfg.top_k
        self.cap_factor = cfg.capacity_factor
        self.dropless = False                      # Phase 2 fine-tuning routes every token
        D, h = cfg.d_model, cfg.expert_ff

        self.router = nn.Linear(D, self.E, bias=False)
        self.w_gate_up = nn.Parameter(torch.empty(self.E, D, 2 * h))    # all experts stacked -> one bmm
        self.w_down = nn.Parameter(torch.empty(self.E, h, D))
        self.shared = SwiGLU(D, cfg.shared_ff)

        nn.init.normal_(self.w_gate_up, std=0.02)
        nn.init.normal_(self.w_down, std=0.02)

        self.register_buffer("bias", torch.zeros(self.E))               # saved: it is learned state
        self.register_buffer("load", torch.zeros(self.E), persistent=False)
        self.register_buffer("dropped", torch.zeros(()), persistent=False)

    def forward(self, x):
        B, T, D = x.shape
        N, E, K = B * T, self.E, self.K
        flat = x.reshape(N, D)

        # --- routing: sigmoid affinities, biased top-k, normalised gate weights
        scores = torch.sigmoid(F.linear(flat.float(), self.router.weight.float()))      # (N, E)
        _, idx = torch.topk(scores.detach() + self.bias, K, dim=-1)                     # bias steers selection only
        gate = scores.gather(1, idx)
        gate = gate / (gate.sum(-1, keepdim=True) + 1e-9)

        # --- slot assignment: position of every (token, expert) pair inside its expert's buffer
        eid = idx.reshape(-1)                                                           # token-major, (N*K,)
        onehot, pos = _slot_positions(eid, E)

        if self.training and not self.dropless:
            cap = int(math.ceil(N * K / E * self.cap_factor / 8) * 8)
        else:
            cap = N                                                                     # drop-free for eval / generation
        keep = pos < cap
        slot = torch.where(keep, eid * cap + pos, torch.full_like(eid, E * cap))        # overflow -> spare row

        # --- dispatch -> one batched matmul over all experts -> combine
        dt = torch.get_autocast_dtype("cuda") if (torch.is_autocast_enabled() and x.is_cuda) else x.dtype
        xs = flat.to(dt).repeat_interleave(K, dim=0)
        buf = xs.new_zeros(E * cap + 1, D).index_put((slot,), xs)
        hid = buf[: E * cap].view(E, cap, D)
        g, u = torch.bmm(hid, self.w_gate_up.to(dt)).chunk(2, dim=-1)
        y = torch.bmm(F.silu(g) * u, self.w_down.to(dt))                                # (E, cap, D)
        y = torch.cat([y.reshape(E * cap, D), y.new_zeros(1, D)], dim=0)
        routed = (y[slot].view(N, K, D) * gate.to(y.dtype).unsqueeze(-1)).sum(1)

        out = (routed.to(x.dtype) + self.shared(flat)).view(B, T, D)

        if not self.training:
            return out, x.new_zeros((), dtype=torch.float32)

        # --- bookkeeping for balancing (no gradient) and the small sequence-wise safety loss
        with torch.no_grad():
            self.load += onehot.sum(0).float()
            self.dropped += (~keep).sum().float()
        p = (scores / (scores.sum(-1, keepdim=True) + 1e-9)).view(B, T, E).mean(1)      # mean affinity per sequence
        f = E * onehot.view(B, T * K, E).float().mean(1)                                # share of assignments per expert
        aux = (f * p).sum(-1).mean()
        return out, aux


# ---------------------------------------------------------------------------
# transformer block, MTP head, full model
# ---------------------------------------------------------------------------
class Block(nn.Module):
    def __init__(self, cfg, use_moe):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = MLA(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.use_moe = use_moe
        self.ffn = MoE(cfg) if use_moe else SwiGLU(cfg.d_model, cfg.d_ff_dense)

    def forward(self, x, cos, sin, past=None, key_mask=None):
        a, present = self.attn(self.attn_norm(x), cos, sin, past, key_mask)
        x = x + a
        if self.use_moe:
            f, aux = self.ffn(self.ffn_norm(x))
        else:
            f, aux = self.ffn(self.ffn_norm(x)), x.new_zeros((), dtype=torch.float32)
        return x + f, present, aux


class MTPHead(nn.Module):
    """Predicts token t+2 from (trunk hidden state at t, embedding of token t+1)."""

    def __init__(self, cfg):
        super().__init__()
        self.norm_h = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.norm_e = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.proj = nn.Linear(2 * cfg.d_model, cfg.d_model, bias=False)
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = MLA(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg.d_model, cfg.mtp_ff)

    def forward(self, h, next_emb, cos, sin):
        z = self.proj(torch.cat([self.norm_h(h), self.norm_e(next_emb)], dim=-1))
        a, _ = self.attn(self.attn_norm(z), cos, sin)
        z = z + a
        return z + self.ffn(self.ffn_norm(z))


class MoEGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg, use_moe=i >= cfg.n_dense_layers) for i in range(cfg.n_layer)])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mtp_head = MTPHead(cfg) if cfg.mtp else None

        cos, sin = build_rope_cache(cfg.qk_rope, cfg.max_seq_len, cfg.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        scale = 0.02 / math.sqrt(2 * cfg.n_layer)                       # GPT-2 style residual-branch scaling
        for blk in self.blocks:
            nn.init.normal_(blk.attn.out.weight, std=scale)
            if blk.use_moe:
                nn.init.normal_(blk.ffn.w_down, std=scale)
                nn.init.normal_(blk.ffn.shared.down.weight, std=scale)
            else:
                nn.init.normal_(blk.ffn.down.weight, std=scale)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    # ---- bookkeeping -------------------------------------------------------
    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def active_params(self):
        """Parameters touched per token: everything except the routed experts that were not picked."""
        total = self.num_params()
        routed = sum(b.ffn.w_gate_up.numel() + b.ffn.w_down.numel() for b in self.blocks if b.use_moe)
        inactive = routed * (1 - self.cfg.top_k / self.cfg.n_routed)
        mtp = sum(p.numel() for p in self.mtp_head.parameters()) if self.mtp_head is not None else 0
        return int(total - inactive - mtp)                              # MTP is training-only

    @torch.no_grad()
    def update_balance(self, speed):
        """
        Aux-loss-free balancing step (call once per optimizer step on every rank):
        push each expert's routing bias toward the mean load, then reset the counters.
        Returns (max load violation, fraction of assignments dropped) for logging.
        """
        moes = [b.ffn for b in self.blocks if b.use_moe]
        load = torch.stack([m.load for m in moes])
        dropped = torch.stack([m.dropped for m in moes])
        if dist.is_available() and dist.is_initialized():
            dist.all_reduce(load)
            dist.all_reduce(dropped)
        mean = load.mean(dim=-1, keepdim=True)
        if speed > 0:
            step = speed * torch.sign(mean - load)                      # under-used experts get a boost
            for m, s in zip(moes, step):
                m.bias += s
        maxvio = ((load - mean).abs().amax(dim=-1) / mean.squeeze(-1).clamp(min=1)).mean()
        drop_frac = dropped.sum() / load.sum().clamp(min=1)
        for m in moes:
            m.load.zero_()
            m.dropped.zero_()
        return maxvio, drop_frac

    def hidden_states(self, idx):
        """Final-layer hidden states (B, T, d_model) - what a Phase 2-7 task head sits on top of."""
        B, T = idx.shape
        cos = self.rope_cos[:T]
        sin = self.rope_sin[:T]
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x, _, _ = blk(x, cos, sin)
        return self.final_norm(x)

    # ---- forward -----------------------------------------------------------
    def forward(self, idx, targets=None, caches=None, start_pos=0, key_mask=None, last_only=False):
        """
        Training:   model(idx, targets) -> (ce, mtp_ce, seq_aux, z_loss)
        Inference:  model(idx, caches=..., ...) -> (logits, new_caches)
        """
        B, T = idx.shape
        cos = self.rope_cos[start_pos:start_pos + T]
        sin = self.rope_sin[start_pos:start_pos + T]

        x = self.tok_emb(idx)
        new_caches, aux_total = [], x.new_zeros((), dtype=torch.float32)
        for i, blk in enumerate(self.blocks):
            past = caches[i] if caches is not None else None
            x, present, aux = blk(x, cos, sin, past, key_mask)
            new_caches.append(present)
            aux_total = aux_total + aux

        hidden = x
        x = self.final_norm(x)
        if last_only:
            x = x[:, -1:, :]
        logits = F.linear(x, self.tok_emb.weight)

        if targets is None:
            return logits, new_caches

        V = logits.size(-1)
        logits = logits.float()
        ce = F.cross_entropy(logits.view(-1, V), targets.reshape(-1))
        z = torch.logsumexp(logits, dim=-1).pow(2).mean()

        mtp_ce = ce.new_zeros(())
        if self.mtp_head is not None and self.training:
            z2 = self.mtp_head(hidden, self.tok_emb(targets), cos, sin)         # embedding of token t+1
            lg2 = F.linear(self.final_norm(z2), self.tok_emb.weight).float()
            shifted = torch.cat([targets[:, 1:], torch.full_like(targets[:, :1], -100)], dim=1)   # token t+2
            mtp_ce = F.cross_entropy(lg2.view(-1, V), shifted.reshape(-1), ignore_index=-100)
        return ce, mtp_ce, aux_total / max(1, len(self.blocks) - self.cfg.n_dense_layers), z


# ---------------------------------------------------------------------------
# Generation (same batched, left-padded KV-cache sampler as arch 3)
# ---------------------------------------------------------------------------
def _sample(logits, temperature, top_k, top_p):
    logits = logits / max(temperature, 1e-5)
    k = min(top_k, logits.size(-1)) if top_k > 0 else logits.size(-1)
    vals, idx = torch.topk(logits, k, dim=-1)
    probs = F.softmax(vals, dim=-1)
    if top_p < 1.0:
        before = probs.cumsum(-1) - probs
        vals = vals.masked_fill(before >= top_p, float("-inf"))
        probs = F.softmax(vals, dim=-1)
    return idx.gather(-1, torch.multinomial(probs, 1)).squeeze(-1)


@torch.no_grad()
def generate(model, prompts, tokenizer, max_new_tokens=384, target_words=128,
             temperature=0.7, top_k=50, top_p=0.92, check_every=8, suppress_eos=True):
    """prompts: list of token-id lists. Returns the generated continuations as strings."""
    model.eval()
    dev = next(model.parameters()).device
    B = len(prompts)
    L = max(len(p) for p in prompts)
    assert L + max_new_tokens <= model.cfg.max_seq_len, "prompt + continuation exceeds context"
    pad, eos = tokenizer.pad_id, tokenizer.eos_id

    ids = torch.full((B, L), pad, dtype=torch.long)
    valid = torch.zeros((B, L), dtype=torch.bool)
    for r, p in enumerate(prompts):
        ids[r, L - len(p):] = torch.tensor(p, dtype=torch.long)
        valid[r, L - len(p):] = True
    ids, valid = ids.to(dev), valid.to(dev)

    causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=dev))
    mask = (causal[None, None] & valid[:, None, None, :]) | torch.eye(L, dtype=torch.bool, device=dev)[None, None]
    logits, caches = model(ids, caches=None, start_pos=0, key_mask=mask, last_only=True)

    out = torch.empty((B, 0), dtype=torch.long, device=dev)
    finished = torch.zeros(B, dtype=torch.bool, device=dev)
    for step in range(max_new_tokens):
        step_logits = logits[:, -1].float()
        step_logits[:, pad] = float("-inf")
        if suppress_eos:                          # the task wants a full-length continuation: stop on word count only
            step_logits[:, eos] = float("-inf")
        nxt = _sample(step_logits, temperature, top_k, top_p)
        nxt = torch.where(finished, torch.full_like(nxt, pad), nxt)
        out = torch.cat([out, nxt[:, None]], dim=1)
        finished |= nxt == eos

        if (step + 1) % check_every == 0:
            for r in torch.nonzero(~finished).flatten().tolist():
                if len(tokenizer.decode(out[r].tolist()).split()) >= target_words:
                    finished[r] = True
        if bool(finished.all()):
            break

        valid = torch.cat([valid, torch.ones((B, 1), dtype=torch.bool, device=dev)], dim=1)
        logits, caches = model(nxt[:, None], caches=caches, start_pos=L + step,
                               key_mask=valid[:, None, None, :], last_only=True)

    texts = []
    for r in range(B):
        toks = out[r].tolist()
        if eos in toks:
            toks = toks[:toks.index(eos)]
        toks = [t for t in toks if t != pad]
        texts.append(" ".join(tokenizer.decode(toks).split()[:target_words]))
    return texts

Writing /kaggle/working/phase6_arch4/phase1_model.py


In [5]:
%%writefile /kaggle/working/phase6_arch4/hub_files.py
"""Finds Phase-1 files: an attached Kaggle dataset first, then a local cache, then Hugging Face."""
import os
import re

from huggingface_hub import hf_hub_download

KAGGLE_INPUT = "/kaggle/input"


def _squash(text):
    return re.sub(r"[^a-z0-9]", "", text.lower())


def find_in_kaggle_input(filename, hint):
    """Only accepts a file whose path mentions the architecture (e.g. 'arch3', 'arch-3', 'Arch_3'),
    because arch 2, 3 and 4 all call their weights model_final.pt."""
    if not os.path.isdir(KAGGLE_INPUT):
        return None
    hits = []
    for root, _, files in os.walk(KAGGLE_INPUT):
        if filename in files:
            path = os.path.join(root, filename)
            if _squash(hint) in _squash(path):
                hits.append(path)
    return sorted(hits)[0] if hits else None


def fetch_from_hub(repo, filename, local_dir):
    try:
        return hf_hub_download(repo, filename, local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def locate(filenames, repos, hint, local_dir):
    """Names are tried in order of preference. Returns (path, where it came from)."""
    for name in filenames:
        path = find_in_kaggle_input(os.path.basename(name), hint)
        if path:
            return path, "kaggle input"
    for name in filenames:
        for repo in repos:
            cached = os.path.join(local_dir, _squash(repo), name)
            if os.path.exists(cached):
                return cached, f"cache of hf://{repo}"
            path = fetch_from_hub(repo, name, os.path.join(local_dir, _squash(repo)))
            if path:
                return path, f"hf://{repo}"
    raise FileNotFoundError(f"None of {filenames} found under /kaggle/input (path must mention '{hint}') or in "
                            f"{repos}. Attach the Phase-1 weights as a Kaggle dataset, or check HF_TOKEN and the repo names.")

Writing /kaggle/working/phase6_arch4/hub_files.py


In [6]:
%%writefile /kaggle/working/phase6_arch4/gen_core.py
"""
Phase 6 (abstractive summarization) machinery shared by all four architectures:
source selection, tokenized arrays, adapters, the decoder-only summarizer wrapper, decoding with length control and
redundancy control, ROUGE, and the submission writer. Each architecture's lm.py supplies the model-specific pieces.
"""
import os
import json
import math
from multiprocessing import Pool

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IGNORE = -100


# ------------------------------------------------------------------------------------------- data
def load_split(path, with_summary):
    with open(path, encoding="utf-8") as f:
        rows = json.load(f)
    ids = [r["id"] for r in rows]
    docs = [r["document"] for r in rows]
    sums = [r["summary"] for r in rows] if with_summary else None
    return ids, docs, sums


_tok = None
_cfg = None


def _init_worker(cfg_dict):
    global _tok, _cfg
    import lm as LM
    _tok = LM.load_tokenizer()
    _cfg = cfg_dict


def select_source(ids, budget, head_frac, n_spread):
    """Keep the opening of the judgment (parties, facts) and evenly spaced pieces of the rest, ending at the last
    token - so the whole case is represented even when it is 50 times longer than the budget."""
    if len(ids) <= budget:
        return ids
    head = int(budget * head_frac)
    piece = (budget - head) // n_spread
    rest = len(ids) - head
    starts = [head + round(i * (rest - piece) / max(1, n_spread - 1)) for i in range(n_spread)]
    out = ids[:head]
    for s in starts:
        out = out + ids[s:s + piece]
    return out


def encode_example(pair):
    doc, summ = pair
    src = select_source(_tok.encode(doc[:_cfg["char_cap"]]), _cfg["src_len"], _cfg["head_frac"], _cfg["n_spread"])
    tgt = _tok.encode(summ)[:_cfg["tgt_len"]] if summ is not None else []
    return src, tgt, len(doc.split()), (len(summ.split()) if summ is not None else 0)


def prepare_split(docs, sums, cfg_dict, workers):
    pairs = list(zip(docs, sums if sums is not None else [None] * len(docs)))
    with Pool(workers, initializer=_init_worker, initargs=(cfg_dict,)) as pool:
        res = pool.map(encode_example, pairs, chunksize=16)
    S, T = cfg_dict["src_len"], cfg_dict["tgt_len"]
    src = np.full((len(res), S), cfg_dict["pad"], dtype=np.int32)
    tgt = np.full((len(res), T), cfg_dict["pad"], dtype=np.int32)
    src_len = np.zeros(len(res), dtype=np.int32)
    tgt_len = np.zeros(len(res), dtype=np.int32)
    doc_words = np.zeros(len(res), dtype=np.int32)
    sum_words = np.zeros(len(res), dtype=np.int32)
    for i, (s, t, dw, sw) in enumerate(res):
        src[i, :len(s)], src_len[i] = s, len(s)
        tgt[i, :len(t)], tgt_len[i] = t, len(t)
        doc_words[i], sum_words[i] = dw, sw
    return {"src": src, "src_len": src_len, "tgt": tgt, "tgt_len": tgt_len, "doc_words": doc_words, "sum_words": sum_words}


# ----------------------------------------------------------------------------------- adaptation
class LoRALinear(nn.Module):
    """Hu et al. 2021: W x + (alpha / r) B A x, B = 0 at the start."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class LoRMALinear(nn.Module):
    """Bihany, Patel & Modi 2025 (LoRMA+): h = (alpha/r B A + I) W0 x, computed as W0x -> A -> B."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + F.linear(F.linear(self.drop(h), self.A), self.B) * self.scale
        return h + self.base.bias if self.base.bias is not None else h


class Adapter(nn.Module):
    """Bottleneck adapter on a sub-layer's output: h + up(gelu(down(h))), up = 0 at the start.
    Houlsby et al. 2019 put one after attention and one after the feed-forward block;
    Pfeiffer et al. 2021 keep only the feed-forward one."""

    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        d = base.out_features
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        h = self.base(x)
        return h + self.up(F.gelu(self.down(h)))


def wrap_linears(root, names, make, skip=("router",)):
    n = 0
    for parent in list(root.modules()):
        for name, child in list(parent.named_children()):
            if name in skip:
                continue
            if name in names and isinstance(child, nn.Linear):
                setattr(parent, name, make(child))
                n += 1
    return n


def configure_method(model, method, cfg):
    if method in ("full", "kd"):
        for p in model.parameters():
            p.requires_grad = True
        return "all weights trainable"
    for p in model.parameters():
        p.requires_grad = False
    if method == "lora":
        n = wrap_linears(model, cfg.ADAPTER_TARGETS, lambda m: LoRALinear(m, cfg.ADAPTER_RANK, cfg.LORA_ALPHA, cfg.ADAPTER_DROPOUT))
        return f"LoRA on {n} attention projections (r={cfg.ADAPTER_RANK})"
    if method == "lorma":
        n = wrap_linears(model, cfg.ADAPTER_TARGETS, lambda m: LoRMALinear(m, cfg.ADAPTER_RANK, cfg.LORMA_ALPHA, cfg.ADAPTER_DROPOUT))
        return f"LoRMA on {n} attention projections (r={cfg.ADAPTER_RANK})"
    if method == "houlsby":
        n = wrap_linears(model, cfg.HOULSBY_SITES, lambda m: Adapter(m, cfg.ADAPTER_BOTTLENECK))
        return f"{n} Houlsby adapters (after attention and feed-forward outputs, size {cfg.ADAPTER_BOTTLENECK})"
    if method == "pfeiffer":
        n = wrap_linears(model, cfg.PFEIFFER_SITES, lambda m: Adapter(m, cfg.ADAPTER_BOTTLENECK))
        return f"{n} Pfeiffer adapters (after the feed-forward output only, size {cfg.ADAPTER_BOTTLENECK})"
    if method == "prompt":
        # Lester et al. 2021: learned virtual tokens in front of the source, initialised from real token embeddings
        g = torch.Generator().manual_seed(cfg.SEED)
        w = model.tok_emb.weight
        pick = torch.randint(100, w.size(0) - 100, (cfg.PROMPT_LEN,), generator=g)
        model.prompt = nn.Parameter(w[pick].detach().clone())
        return f"{cfg.PROMPT_LEN} soft-prompt vectors in front of every source"
    raise ValueError(method)


def count_params(model):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)


def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().float().half() for k, v in model.state_dict().items() if k in names}


def build_optimizer(model, method, cfg):
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name == "prompt":
            lr = cfg.PROMPT_LR
        elif method in ("full", "kd"):
            lr = cfg.FULL_LR
        else:
            lr = cfg.ADAPTER_LR
        wd = 0.0 if (p.ndim < 2 or name == "prompt") else cfg.WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()],
                             betas=(0.9, 0.98), eps=1e-6)


# ------------------------------------------------------------------- decoder-only summarizer
def decoder_batch(src, src_len, tgt, tgt_len, sep_id, pad_id, eos_id):
    """Right-padded [source] <sep> [summary] (<eos> if the summary was not cut) and next-token labels that are
    only set where the next token belongs to the summary."""
    rows, labels = [], []
    for s, sl, t, tl in zip(src, src_len, tgt, tgt_len):
        summary = list(t[:tl]) + ([eos_id] if tl < len(t) else [])
        seq = list(s[:sl]) + [sep_id] + summary
        lab = [IGNORE] * sl + summary + [IGNORE]
        rows.append(seq)
        labels.append(lab[:len(seq)])
    L = max(map(len, rows))
    ids = torch.full((len(rows), L), pad_id, dtype=torch.long)
    lab = torch.full((len(rows), L), IGNORE, dtype=torch.long)
    for i, (r, l) in enumerate(zip(rows, labels)):
        ids[i, :len(r)] = torch.tensor(r)
        lab[i, :len(l)] = torch.tensor(l)
    return ids, lab


class DecoderLM(nn.Module):
    """Wraps a Phase-1 decoder-only model. Subclasses (lm.py) provide add_positions() and run_blocks()."""
    is_encoder_decoder = False

    def __init__(self):
        super().__init__()
        self.prompt = None

    def _embed(self, ids, valid):
        x = self.tok_emb(ids)
        if self.prompt is not None:
            P = self.prompt.size(0)
            x = torch.cat([self.prompt.to(x.dtype).unsqueeze(0).expand(x.size(0), -1, -1), x], dim=1)
            valid = torch.cat([valid.new_ones(valid.size(0), P), valid], dim=1)
        pos = (valid.long().cumsum(1) - 1).clamp(min=0)
        return x, valid, pos

    def teacher_forced(self, batch, device):
        ids, labels = batch
        ids, labels = ids.to(device), labels.to(device)
        x, valid, pos = self._embed(ids, torch.ones_like(ids, dtype=torch.bool))
        x = self.add_positions(x, pos)
        h, _ = self.run_blocks(x, pos, None, None)
        h = self.final_norm(h)[:, h.size(1) - ids.size(1):]
        keep = labels != IGNORE
        logits = F.linear(h[keep], self.tok_emb.weight)
        return logits.float(), labels[keep]

    def make_batch(self, src, src_len, tgt, tgt_len, tok):
        return decoder_batch(src, src_len, tgt, tgt_len, tok.sep_id, tok.pad_id, tok.eos_id)

    @torch.no_grad()
    def generate(self, sources, min_new, max_new, tok, cfg, device):
        """sources: list of token lists. Left-padded batch; every row stops on its own length budget."""
        B = len(sources)
        rows = [list(s) + [tok.sep_id] for s in sources]
        L = max(map(len, rows))
        ids = torch.full((B, L), tok.pad_id, dtype=torch.long)
        valid = torch.zeros((B, L), dtype=torch.bool)
        for i, r in enumerate(rows):
            ids[i, L - len(r):] = torch.tensor(r)
            valid[i, L - len(r):] = True
        ids, valid = ids.to(device), valid.to(device)
        x, valid, pos = self._embed(ids, valid)
        x = self.add_positions(x, pos)
        S = x.size(1)
        causal = torch.tril(torch.ones(S, S, dtype=torch.bool, device=device))
        mask = (causal[None, None] & valid[:, None, None, :]) | torch.eye(S, dtype=torch.bool, device=device)[None, None]
        h, caches = self.run_blocks(x, pos, None, mask)
        logits = F.linear(self.final_norm(h[:, -1:]), self.tok_emb.weight)[:, -1].float()
        next_pos = pos[:, -1] + 1

        state = DecodeState(B, logits.size(-1), min_new, max_new, tok, cfg, device)
        while not state.done():
            nxt = state.step(logits)
            valid = torch.cat([valid, torch.ones((B, 1), dtype=torch.bool, device=device)], dim=1)
            x = self.add_positions(self.tok_emb(nxt[:, None]), next_pos[:, None])
            h, caches = self.run_blocks(x, next_pos[:, None], caches, valid[:, None, None, :])
            logits = F.linear(self.final_norm(h), self.tok_emb.weight)[:, -1].float()
            next_pos = next_pos + 1
        return state.outputs()


# ---------------------------------------------------------------------------------- decoding
class DecodeState:
    """Greedy decoding with a length budget per row, a repetition penalty (Keskar et al. 2019) and no repeated
    n-grams - the redundancy controls the summaries need."""

    def __init__(self, B, V, min_new, max_new, tok, cfg, device):
        self.B, self.tok, self.cfg = B, tok, cfg
        self.min_new = torch.tensor(min_new, device=device)
        self.max_new = torch.tensor(max_new, device=device)
        self.seen = torch.zeros((B, V), dtype=torch.bool, device=device)
        self.out = [[] for _ in range(B)]
        self.ngrams = [dict() for _ in range(B)]
        self.finished = torch.zeros(B, dtype=torch.bool, device=device)
        self.t = 0

    def done(self):
        return bool(self.finished.all())

    def step(self, logits):
        cfg, tok = self.cfg, self.tok
        logits = logits.clone()
        logits[:, tok.pad_id] = float("-inf")
        for name in ("unk_id", "bos_id"):                # arch 1's word vocabulary has these; decoding would drop them
            if hasattr(tok, name):
                logits[:, getattr(tok, name)] = float("-inf")
        logits[self.t < self.min_new, tok.eos_id] = float("-inf")
        if cfg.REPETITION_PENALTY != 1.0:
            pen = torch.where(logits > 0, logits / cfg.REPETITION_PENALTY, logits * cfg.REPETITION_PENALTY)
            logits = torch.where(self.seen, pen, logits)
        n = cfg.NO_REPEAT_NGRAM
        if n > 1:
            for i in range(self.B):
                if not self.finished[i] and len(self.out[i]) >= n - 1:
                    banned = self.ngrams[i].get(tuple(self.out[i][-(n - 1):]))
                    if banned:
                        logits[i, list(banned)] = float("-inf")
        nxt = logits.argmax(-1)
        nxt = torch.where(self.finished, torch.full_like(nxt, tok.pad_id), nxt)
        for i, t in enumerate(nxt.tolist()):
            if self.finished[i]:
                continue
            seq = self.out[i]
            seq.append(t)
            if n > 1 and len(seq) >= n:
                self.ngrams[i].setdefault(tuple(seq[-n:-1]), set()).add(seq[-1])
        self.seen[torch.arange(self.B), nxt] = True
        self.t += 1
        self.finished |= (nxt == tok.eos_id) | (self.t >= self.max_new)
        return nxt

    def outputs(self):
        return [[t for t in seq if t not in (self.tok.eos_id, self.tok.pad_id)] for seq in self.out]


def length_budget(doc_words, cfg, tokens_per_word, ceiling):
    """Summary length follows document length (the training median ratio), clipped to sensible bounds."""
    words = np.clip(np.asarray(doc_words) / cfg.LENGTH_RATIO, cfg.MIN_SUMMARY_WORDS, cfg.MAX_SUMMARY_WORDS)
    target = words * tokens_per_word
    max_new = np.minimum(np.ceil(target * (1 + cfg.LENGTH_SLACK)), ceiling).astype(int)
    min_new = np.minimum(np.floor(target * (1 - cfg.LENGTH_SLACK)), max_new - 1).astype(int)
    return min_new.tolist(), max_new.tolist()


def tidy(text, sentence_case):
    text = " ".join(text.split())
    if sentence_case:                                    # arch 1 generates lower-case words
        out, cap = [], True
        for w in text.split(" "):
            out.append(w[:1].upper() + w[1:] if cap and w[:1].isalpha() else w)
            cap = w.endswith((".", "?", "!"))
        text = " ".join(out)
    return text


# ------------------------------------------------------------------------------------------ ROUGE
def _lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]


def rouge(ref, hyp):
    r, h = ref.lower().split(), hyp.lower().split()
    if not r or not h:
        return 0.0, 0.0
    from collections import Counter
    overlap = sum((Counter(r) & Counter(h)).values())
    p1, r1 = overlap / len(h), overlap / len(r)
    f1 = 2 * p1 * r1 / (p1 + r1) if p1 + r1 else 0.0
    lcs = _lcs(r, h)
    pl, rl = lcs / len(h), lcs / len(r)
    fl = 2 * pl * rl / (pl + rl) if pl + rl else 0.0
    return f1, fl


def write_submission(ids, summaries, out_dir, zip_name):
    import zipfile
    assert len(ids) == len(summaries) == len(set(map(str, ids)))
    payload = [{"id": i, "summary": s if s.strip() else "."} for i, s in zip(ids, summaries)]
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    zip_path = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist() == ["prediction.json"]
    return path, zip_path

Writing /kaggle/working/phase6_arch4/gen_core.py


In [7]:
%%writefile /kaggle/working/phase6_arch4/lm.py
"""Arch 4 (DeepSeek-style MoE decoder) as a summarizer: reads [source] <eos> [summary] and writes the summary."""
import os

import torch
import torch.nn as nn

import config as C
from hub_files import locate, fetch_from_hub  # noqa: F401  (fetch_from_hub is used by run_sum)
from phase1_model import MoEGPT, ModelConfig
from tokenizer import NerTokenizer
from gen_core import DecoderLM


def load_tokenizer():
    path, _ = locate(C.TOKENIZER_FILES, C.TOKENIZER_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    tok = NerTokenizer.load(path)
    tok.sep_id = tok.eos_id
    return tok


class Arch4LM(DecoderLM):
    def __init__(self, gpt, keep_layers):
        super().__init__()
        self.cfg = gpt.cfg
        self.tok_emb = gpt.tok_emb
        self.layers = nn.ModuleList(gpt.blocks if keep_layers is None else [gpt.blocks[i] for i in keep_layers])
        self.final_norm = gpt.final_norm
        self.register_buffer("rope_cos", gpt.rope_cos.clone(), persistent=False)
        self.register_buffer("rope_sin", gpt.rope_sin.clone(), persistent=False)
        self.max_len = gpt.cfg.max_seq_len
        for blk in self.layers:
            if blk.use_moe:
                blk.ffn.dropless = True          # every token is routed while fine-tuning; routing biases stay frozen

    def add_positions(self, x, pos):
        return x

    def run_blocks(self, x, pos, caches, key_mask):
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        new = []
        for i, blk in enumerate(self.layers):
            x, present, _ = blk(x, cos, sin, None if caches is None else caches[i], key_mask)
            new.append(present)
        return x, new


def build_lm(keep_layers=None):
    path, source = locate(C.WEIGHT_FILES, C.PHASE1_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    blob = torch.load(path, map_location="cpu")
    cfg = dict(blob["model_cfg"], mtp=False)             # the multi-token head is a pre-training-only extra
    gpt = MoEGPT(ModelConfig(**cfg))
    gpt.load_state_dict({k: v.float() for k, v in blob["model"].items() if not k.startswith("mtp_head.")})
    if os.environ.get("P2_VERBOSE", "1") == "1":
        print(f"Phase-1 arch 4 weights: {path} ({source})", flush=True)
    return Arch4LM(gpt, keep_layers)


def student_layers():
    return list(C.STUDENT_LAYERS)

Writing /kaggle/working/phase6_arch4/lm.py


In [8]:
%%writefile /kaggle/working/phase6_arch4/run_sum.py
"""
One Phase-6 experiment = one call:

    python run_sum.py --method lorma

Trains for a fixed wall-clock budget (the best checkpoint is chosen by teacher-forced loss on held-out documents),
generates summaries for some held-out documents to measure ROUGE-1 / ROUGE-L and length, then summarizes the test set
and pushes checkpoint, results and submission zip to Hugging Face.
"""
import os
import sys
import json
import math
import time
import random
import argparse

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as C
import lm as LM
from gen_core import (configure_method, count_params, trainable_state, build_optimizer, length_budget, tidy, rouge,
                      write_submission)


def log(*a):
    print(*a, flush=True)


def build_model(method, keep_layers=None):
    model = LM.build_lm(keep_layers)
    note = configure_method(model, method, C)
    return model, note


def load_finetuned(method, device):
    path = os.path.join(C.OUT_DIR, method, f"sum_{method}.pt")
    if not os.path.exists(path):
        path = LM.fetch_from_hub(C.SUM_REPO, f"{method}/sum_{method}.pt", os.path.join(C.OUT_DIR, "_hub"))
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"No fine-tuned '{method}' model locally or on {C.SUM_REPO}; run '{method}' first.")
    blob = torch.load(path, map_location="cpu")
    model, _ = build_model(method, blob.get("keep_layers"))
    _, unexpected = model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    assert not unexpected, unexpected
    return model.to(device).eval()


def push(files, message):
    if not os.environ.get("HF_TOKEN"):
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(C.SUM_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
        for local, remote in files:
            api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=C.SUM_REPO,
                            token=os.environ["HF_TOKEN"], commit_message=message)
        log(f"pushed to {C.SUM_REPO}: {message}")
    except Exception as e:
        log(f"HF push failed ({type(e).__name__}: {e}) - the files are still in {C.OUT_DIR}")


def load_npz(name):
    z = np.load(os.path.join(C.DATA_DIR, f"{name}.npz"))
    return {k: z[k] for k in z.files}


@torch.no_grad()
def dev_loss(model, d, idx, tok, device):
    model.eval()
    total, n = 0.0, 0
    for b in range(0, len(idx), C.MICRO_DOCS):
        j = idx[b:b + C.MICRO_DOCS]
        batch = model.make_batch(d["src"][j], d["src_len"][j], d["tgt"][j], d["tgt_len"][j], tok)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            logits, gold = model.teacher_forced(batch, device)
        total += F.cross_entropy(logits, gold, reduction="sum").item()
        n += gold.numel()
    return total / max(1, n)


def summarize(model, d, idx, tok, device, tokens_per_word):
    srcs = [d["src"][i][:d["src_len"][i]].tolist() for i in idx]
    ceiling = C.MAX_NEW_TOKENS
    min_new, max_new = length_budget(d["doc_words"][idx], C, tokens_per_word, ceiling)
    out = [None] * len(idx)
    order = np.argsort([len(s) for s in srcs])                    # similar lengths together -> little padding
    for b in range(0, len(order), C.GEN_BATCH):
        part = order[b:b + C.GEN_BATCH]
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            toks = model.generate([srcs[i] for i in part], [min_new[i] for i in part], [max_new[i] for i in part],
                                  tok, C, device)
        for i, t in zip(part, toks):
            out[i] = tidy(tok.decode(t), C.SENTENCE_CASE)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True, choices=["full", "lorma", "lora", "houlsby", "pfeiffer", "prompt", "kd"])
    method = ap.parse_args().method
    random.seed(C.SEED)
    np.random.seed(C.SEED)
    torch.manual_seed(C.SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = os.path.join(C.OUT_DIR, method)
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()

    tok = LM.load_tokenizer()
    train, test = load_npz("train"), load_npz("test")
    meta = json.load(open(os.path.join(C.DATA_DIR, "meta.json")))
    if C.LENGTH_RATIO == "auto":
        C.LENGTH_RATIO = meta["length_ratio"]          # median document/summary word ratio of the training set
    n = len(train["src_len"])
    perm = np.random.RandomState(C.SEED).permutation(n)
    dev_idx, tr_idx = np.sort(perm[:C.DEV_DOCS]), perm[C.DEV_DOCS:]
    if C.TRAIN_DOC_LIMIT:
        tr_idx = tr_idx[:C.TRAIN_DOC_LIMIT]
    loss_idx, gen_idx = dev_idx[:C.DEV_LOSS_DOCS], dev_idx[:C.DEV_GEN_DOCS]
    log(f"[{method}] train {len(tr_idx):,} docs | held-out {len(dev_idx)} | test {len(test['src_len'])} | "
        f"source {int(train['src_len'].mean())} tokens, summary {int(train['tgt_len'].mean())} tokens on average")

    keep = LM.student_layers() if method == "kd" else None
    model, note = build_model(method, keep)
    model.to(device)
    total, trainable = count_params(model)
    log(f"[{method}] {note} | params {total / 1e6:.1f}M, trainable {trainable / 1e6:.2f}M | budget {C.TRAIN_MINUTES} min")

    teacher = None
    if method == "kd":
        teacher = load_finetuned(C.KD_TEACHER, device)
        log(f"[kd] teacher = fine-tuned '{C.KD_TEACHER}' ({count_params(teacher)[0] / 1e6:.1f}M) -> student keeping "
            f"layers {keep} ({total / 1e6:.1f}M); it learns the teacher's next-token distributions (word-level KD)")

    optimizer = build_optimizer(model, method, C)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    docs_per_step = C.MICRO_DOCS * C.GRAD_ACCUM
    step_cap = max(1, C.MAX_EPOCHS * len(tr_idx) // docs_per_step)
    budget = C.TRAIN_MINUTES * 60
    order = np.random.RandomState(C.SEED).permutation(tr_idx)
    cursor, step, seen, running = 0, 0, 0, 0.0
    best, history = {"loss": float("inf"), "step": 0}, []
    t_train = time.time()

    def progress():
        return max(step / step_cap, (time.time() - t_train) / budget)

    while progress() < 1.0:
        warm = min(1.0, (step + 1) / C.WARMUP_STEPS)
        lr_scale = warm * (C.MIN_LR_RATIO + (1 - C.MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(1.0, progress()))))
        for g, lr in zip(optimizer.param_groups, base_lrs):
            g["lr"] = lr * lr_scale
        model.train()
        for _ in range(C.GRAD_ACCUM):
            if cursor + C.MICRO_DOCS > len(order):
                order = np.random.RandomState(C.SEED + step).permutation(tr_idx)
                cursor = 0
            j = np.sort(order[cursor:cursor + C.MICRO_DOCS])
            cursor += C.MICRO_DOCS
            batch = model.make_batch(train["src"][j], train["src_len"][j], train["tgt"][j], train["tgt_len"][j], tok)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                logits, gold = model.teacher_forced(batch, device)
                loss = F.cross_entropy(logits, gold, label_smoothing=C.LABEL_SMOOTHING)
                if teacher is not None:
                    with torch.no_grad():
                        t_logits, _ = teacher.teacher_forced(batch, device)
                    T = C.KD_TEMPERATURE
                    kl = F.kl_div(F.log_softmax(logits / T, -1), F.softmax(t_logits.float() / T, -1),
                                  reduction="batchmean") * T * T
                    loss = C.KD_ALPHA * loss + (1 - C.KD_ALPHA) * kl
            scaler.scale(loss / C.GRAD_ACCUM).backward()
            running += loss.item()
            seen += 1
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([q for q in model.parameters() if q.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        step += 1

        if step % C.EVAL_EVERY_STEPS == 0 or progress() >= 1.0:
            dl = dev_loss(model, train, loss_idx, tok, device)
            history.append({"step": step, "docs_seen": step * docs_per_step, "train_loss": running / max(1, seen),
                            "dev_loss": dl, "minutes": (time.time() - t0) / 60})
            log(f"[{method}] step {step} | {step * docs_per_step:,} docs seen | loss {running / max(1, seen):.4f} | "
                f"held-out loss {dl:.4f} (ppl {math.exp(min(dl, 20)):.1f}) | {100 * min(1.0, progress()):.0f}% of budget")
            running, seen = 0.0, 0
            if dl < best["loss"]:
                best = {"loss": dl, "step": step}
                torch.save({"method": method, "keep_layers": keep, "weights": trainable_state(model), "step": step},
                           os.path.join(out_dir, f"sum_{method}.pt"))

    # --------------------------------------------- best checkpoint: summaries for held-out docs, then the test set
    blob = torch.load(os.path.join(out_dir, f"sum_{method}.pt"), map_location="cpu")
    model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    model.eval()
    del teacher
    if device.type == "cuda":
        torch.cuda.empty_cache()

    with open(os.path.join(C.DATA_DIR, "train_summaries.json")) as f:
        references = json.load(f)
    t_gen = time.time()
    dev_out = summarize(model, train, gen_idx, tok, device, meta["tokens_per_word"])
    gen_seconds = time.time() - t_gen
    pairs = [rouge(references[int(i)], h) for i, h in zip(gen_idx, dev_out)]
    r1 = float(np.mean([p[0] for p in pairs]))
    rl = float(np.mean([p[1] for p in pairs]))
    words = [len(h.split()) for h in dev_out]
    ref_words = [len(references[int(i)].split()) for i in gen_idx]

    test_out = summarize(model, test, np.arange(len(test["src_len"])), tok, device, meta["tokens_per_word"])
    with open(os.path.join(C.DATA_DIR, "test_ids.json")) as f:
        test_ids = json.load(f)
    _, zip_path = write_submission(test_ids, test_out, out_dir, f"submission_{C.ARCH}_{method}.zip")
    with open(os.path.join(out_dir, "dev_samples.json"), "w") as f:
        json.dump([{"reference": references[int(i)][:1500], "generated": h[:1500]} for i, h in zip(gen_idx[:5], dev_out[:5])],
                  f, indent=1, ensure_ascii=False)

    summary = {
        "arch": C.ARCH, "method": method, "best_step": best["step"], "heldout_loss": best["loss"],
        "heldout_ppl": math.exp(min(best["loss"], 20)), "dev_rouge1_f1": r1, "dev_rougeL_f1": rl,
        "dev_words_generated": float(np.mean(words)), "dev_words_reference": float(np.mean(ref_words)),
        "test_words_generated": float(np.mean([len(s.split()) for s in test_out])),
        "params_total": total, "params_trainable": trainable,
        "seconds_per_summary": round(gen_seconds / max(1, len(gen_idx)), 2),
        "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if device.type == "cuda" else 0.0,
        "minutes": round((time.time() - t0) / 60, 1), "docs_seen": step * docs_per_step, "history": history,
    }
    with open(os.path.join(out_dir, "results.json"), "w") as f:
        json.dump(summary, f, indent=2)
    log(f"[{method}] DONE | held-out loss {best['loss']:.4f} | ROUGE-1 {r1:.4f} ROUGE-L {rl:.4f} on {len(gen_idx)} held-out docs | "
        f"{np.mean(words):.0f} words generated vs {np.mean(ref_words):.0f} in references -> {zip_path}")

    push([(os.path.join(out_dir, f"sum_{method}.pt"), f"{method}/sum_{method}.pt"),
          (os.path.join(out_dir, "results.json"), f"{method}/results.json"),
          (os.path.join(out_dir, "dev_samples.json"), f"{method}/dev_samples.json"),
          (zip_path, f"{method}/{os.path.basename(zip_path)}")], f"{C.ARCH} {method}: ROUGE-L {rl:.4f}")


if __name__ == "__main__":
    main()

Writing /kaggle/working/phase6_arch4/run_sum.py


## 4. Data: download, select the source, tokenize
Done once with all CPU cores and stored as compact arrays.

In [9]:
import importlib
import config as C
importlib.reload(C)
import lm as LM
import gen_core as G
importlib.reload(G)

for split, folder in (("train", "Train"), ("test", "Test")):
    hf_hub_download(C.HF_DATASET_REPO, f"Phase_6_Summ/{folder}/{split}.json", repo_type="dataset",
                    local_dir=C.DATA_DIR, token=HF_TOKEN)

tok = LM.load_tokenizer()                       # caches the tokenizer before the worker processes start
prep_cfg = {"src_len": C.SRC_LEN, "tgt_len": C.TGT_LEN, "char_cap": C.CHAR_CAP, "head_frac": C.HEAD_FRAC,
            "n_spread": C.N_SPREAD, "pad": tok.pad_id}

ids, docs, sums = G.load_split(C.RAW["train"], with_summary=True)
arrays = G.prepare_split(docs, sums, prep_cfg, C.PREP_WORKERS)
np.savez(os.path.join(C.DATA_DIR, "train.npz"), **arrays)
json.dump(sums, open(os.path.join(C.DATA_DIR, "train_summaries.json"), "w"))
full = arrays["tgt_len"] < C.TGT_LEN
meta = {"tokens_per_word": float(np.median(arrays["tgt_len"][full] / np.maximum(arrays["sum_words"][full], 1))),
        "length_ratio": float(np.median(arrays["doc_words"] / np.maximum(arrays["sum_words"], 1)))}
json.dump(meta, open(os.path.join(C.DATA_DIR, "meta.json"), "w"))
print(f"train: {len(ids):,} docs | source {arrays['src_len'].mean():.0f} tokens | summary {arrays['tgt_len'].mean():.0f} tokens "
      f"({100 * (~full).mean():.0f}% cut at {C.TGT_LEN}) | {meta['tokens_per_word']:.2f} tokens/word | "
      f"document/summary ratio {meta['length_ratio']:.2f}")
del docs, sums, arrays
gc.collect()

ids, docs, _ = G.load_split(C.RAW["test"], with_summary=False)
arrays = G.prepare_split(docs, None, prep_cfg, C.PREP_WORKERS)
np.savez(os.path.join(C.DATA_DIR, "test.npz"), **arrays)
json.dump(ids, open(os.path.join(C.DATA_DIR, "test_ids.json"), "w"))
print(f"test: {len(ids)} docs | source {arrays['src_len'].mean():.0f} tokens | document words median {np.median(arrays['doc_words']):.0f}")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/413k [00:00<?, ?B/s]

train: 7,030 docs | source 992 tokens | summary 727 tokens (31% cut at 1024) | 1.17 tokens/word | document/summary ratio 4.61
test: 100 docs | source 991 tokens | document words median 3171


## 5. Load the Phase-1 weights once
A quick generation from the untouched Phase-1 model checks the whole decoding path and caches the files.

In [10]:
probe = LM.build_lm().eval()
test = np.load(os.path.join(C.DATA_DIR, "test.npz"))
with torch.no_grad():
    out = probe.generate([test["src"][0][:200].tolist()], [10], [20], tok, C, torch.device("cpu"))
print(f"model ready: {sum(p.numel() for p in probe.parameters()) / 1e6:.1f}M parameters | sample continuation: {tok.decode(out[0])!r}")
del probe, test

model_best.pt: reconstructing file:   0%|          |  0.00B /  445MB            

model_best.pt: downloading bytes:           |  0.00B            

Phase-1 arch 4 weights: /kaggle/working/phase6_arch4/phase1/d240357legalarch4moev2/model_best.pt (hf://D-240357/legal-arch4-moe-v2)
model ready: 217.4M parameters | sample continuation: 'Court No. - 1 Case :- CRIMINAL MISC. WRIT PETITION No. - 13972 of 2018'


## 6. Run the experiments
Each experiment is its own process (`run_sum.py --method ...`); with two GPUs, two run at once, so the six take about
three rounds of `TRAIN_MINUTES` + ~15 minutes (training, then summarising 40 held-out and 100 test documents).
`kd` starts as soon as `full` (its teacher) is done. Results, checkpoints and zips are pushed to `D-240357/legal-sum-arch4`.

In [11]:
import subprocess, threading

pending = list(C.METHODS)
done, failed = set(), set()
lock = threading.Lock()


def next_job():
    with lock:
        if "kd" in pending and C.KD_TEACHER in failed:
            pending.remove("kd")
            print(f"skipping kd: its teacher '{C.KD_TEACHER}' failed")
        for m in pending:
            if m != "kd" or C.KD_TEACHER in done or C.KD_TEACHER not in C.METHODS:
                pending.remove(m)
                return m
        return "wait" if pending else None


def worker(gpu):
    while True:
        job = next_job()
        if job is None:
            return
        if job == "wait":
            time.sleep(15)
            continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        proc = subprocess.Popen([sys.executable, "run_sum.py", "--method", job], cwd=WORK_DIR, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(f"[gpu{gpu}] {line}", end="")
        with lock:
            (done if proc.wait() == 0 else failed).add(job)


t0 = time.time()
threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"\nfinished in {(time.time() - t0) / 60:.0f} min | ok: {sorted(done)} | failed: {sorted(failed)}")

[gpu1] [lorma] train 6,880 docs | held-out 150 | test 100 | source 991 tokens, summary 727 tokens on average
[gpu0] [full] train 6,880 docs | held-out 150 | test 100 | source 991 tokens, summary 727 tokens on average
[gpu0] Phase-1 arch 4 weights: /kaggle/working/phase6_arch4/phase1/d240357legalarch4moev2/model_best.pt (cache of hf://D-240357/legal-arch4-moe-v2)
[gpu1] Phase-1 arch 4 weights: /kaggle/working/phase6_arch4/phase1/d240357legalarch4moev2/model_best.pt (cache of hf://D-240357/legal-arch4-moe-v2)
[gpu1] [lorma] LoRMA on 60 attention projections (r=8) | params 218.1M, trainable 0.67M | budget 40 min
[gpu0] [full] all weights trainable | params 217.4M, trainable 217.45M | budget 40 min
[gpu1] [lorma] step 150 | 1,200 docs seen | loss 3.1150 | held-out loss 2.4489 (ppl 11.6) | 13% of budget
[gpu0] [full] step 150 | 1,200 docs seen | loss 3.0208 | held-out loss 2.3678 (ppl 10.7) | 15% of budget
[gpu1] [lorma] step 300 | 2,400 docs seen | loss 3.0108 | held-out loss 2.4214 (ppl 1

## 7. Results
Held-out ROUGE-1 / ROUGE-L, length against the references, and cost. A generated summary is shown next to its reference.

In [12]:
import pandas as pd

rows = []
for path in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "results.json"))):
    r = json.load(open(path))
    rows.append({"method": r["method"], "ROUGE-1": r["dev_rouge1_f1"], "ROUGE-L": r["dev_rougeL_f1"],
                 "held-out loss": r["heldout_loss"], "words (gen)": r["dev_words_generated"],
                 "words (ref)": r["dev_words_reference"], "docs seen": r["docs_seen"],
                 "params (M)": r["params_total"] / 1e6, "trainable %": 100 * r["params_trainable"] / r["params_total"],
                 "s / summary": r["seconds_per_summary"], "minutes": r["minutes"], "peak GPU GB": r["peak_gpu_gb"]})
table = pd.DataFrame(rows).sort_values("ROUGE-L", ascending=False).round(4)
display(table)
best = table.iloc[0]["method"]
sample = json.load(open(os.path.join(C.OUT_DIR, best, "dev_samples.json")))[0]
print(f"--- reference ---\n{sample['reference'][:800]}\n\n--- {best} ---\n{sample['generated'][:800]}")

,method,ROUGE-1,ROUGE-L,held-out loss,words (gen),words (ref),docs seen,params (M),trainable %,s / summary,minutes,peak GPU GB
1,houlsby,0.4306,0.2288,2.3551,534.050,707.625,9672,219.8292,1.0823,7.99,59.9,3.44
0,full,0.4400,0.2286,2.2857,545.900,707.625,7888,217.4500,100.0000,6.66,57.0,7.77
4,lorma,0.4304,0.2262,2.3871,540.475,707.625,9208,218.1166,0.3056,8.80,61.8,3.60
3,lora,0.4282,0.2237,2.3741,542.600,707.625,9200,218.0598,0.2796,8.69,61.5,3.55
2,kd,0.4260,0.2152,2.8864,562.075,707.625,8776,115.6927,100.0000,3.61,49.2,5.73
5,prompt,0.4143,0.2088,2.4801,595.725,707.625,9792,217.4623,0.0057,7.84,59.3,3.35


--- reference ---
As the reversioner of a Hindu widow 's estate one 'R ' instituted suit No. 4 of 1950 in the court of the Civil Judge Jaunpur, for a declaration that he was the 'owner in possession ' of the said estate. 'B ' who claimed to be an heir of the widow was impleaded as a party defendant. The suit was decreed ex parte. 'B ' then filed suit No. 14 of 1956 in the same court against the heirs of 'R ' who died after the, passing of the decree in his suit. In suit No., 14 of 1956 'B ' claimed that as heir of the said widow he was entitled to her stridhuna properties. He averred that in suit No. 4 of 1950 'R ' had arrived at an oral compromise with him promising him 5/16th share in the, whole estate.; that the oral compromise was later reduced into a memorandum; that 'R ' had represented to him that a 

--- houlsby ---
The appellant, a Hindu undivided family, was the owner of certain properties in respect of which the suit was filed. The suit was decreed ex parte. The appellant fi

## 8. Pick a submission
Every experiment wrote `runs/<method>/submission_<arch>_<method>.zip`: a single `prediction.json` at the root holding a
JSON list of `{"id": ..., "summary": ...}` for all 100 test documents. The best on held-out ROUGE-L is also copied to
`/kaggle/working/submission.zip`. Upload the zips exactly as they are.

In [13]:
shutil.copy(glob.glob(os.path.join(C.OUT_DIR, best, "submission_*.zip"))[0], "/kaggle/working/submission.zip")
print(f"best on held-out ROUGE-L: {best} -> /kaggle/working/submission.zip")
for z in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "submission_*.zip"))):
    print("  ", z)

best on held-out ROUGE-L: houlsby -> /kaggle/working/submission.zip
   /kaggle/working/phase6_arch4/runs/full/submission_arch4_full.zip
   /kaggle/working/phase6_arch4/runs/houlsby/submission_arch4_houlsby.zip
   /kaggle/working/phase6_arch4/runs/kd/submission_arch4_kd.zip
   /kaggle/working/phase6_arch4/runs/lora/submission_arch4_lora.zip
   /kaggle/working/phase6_arch4/runs/lorma/submission_arch4_lorma.zip
   /kaggle/working/phase6_arch4/runs/prompt/submission_arch4_prompt.zip
